# 🏎️ F1 Pit Stop — Full Pipeline

**LGB · XGB · CatBoost · RealMLP-A · RealMLP-B → Stacking → Blend**

---

## The key insight

Every notebook in this competition uses tyre life, degradation, 
and race progress. That gets you to ~0.953.

The extra **+0.0001** comes from one question nobody was asking:

> *Does it matter that it's **Hamilton**, at **Bahrain**, in **2021**?*

Yes. Dramatically. Different drivers, teams, and seasons have 
fundamentally different pit strategies — and encoding that 
history directly as a probability estimate gives every model 
in the ensemble a massive shortcut.

This notebook builds a full 5-model pipeline around that idea.
See the **Target Encoding** section below for exactly how it works.

---

## Results

| Model | OOF AUC |
|-------|---------|
| LGB | 0.95241 |
| XGB | 0.95232 |
| CatBoost | 0.95127 |
| RealMLP-A | 0.95260 |
| RealMLP-B | 0.95259 |
| **Stack v8** | **0.95357** |
| **Final blend** | **0.95354** |

---

## What's inside

- **118 tree features + 26 MLP features** (two parallel feature sets)
- **CV Target Encoding** — Driver×Race×Year, Driver×Race, 
  Driver×Compound (leak-free, computed inside each fold)
- **LGB · XGB(Optuna) · CatBoost · RealMLP-A · RealMLP-B**
- **Stacking v8** → hill-climb blend → external blend with 
  top public notebooks
- Historical pit priors from official F1 data (1950–2022)

Runtime: ~50 min on T4×2 GPU

---

## Credits

Pipeline architecture based on 
[v8 by Svanik Kolli](https://www.kaggle.com/code/svanikkolli/f1-lap-by-lap-prediction-engine).
Extended with Driver×Race×Year encoding and additional features.

*If this helped you, an upvote keeps the motivation going 🙏*

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
%%time
import subprocess
for pkg in ["pytabkit", "lightgbm>=4.3.0", "xgboost>=2.0.0", "catboost", "colorama", "optuna"]:
    subprocess.run(["pip", "install", "-q", pkg], check=True)
print("Done.")


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import random, gc, json, glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from colorama import Fore, Style
from importlib.metadata import version
from scipy.stats import rankdata
from scipy.optimize import minimize, differential_evolution
from scipy.special import logit as _logit

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from pytabkit import RealMLP_TD_Classifier

TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e5/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e5/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv'
ORIG_PATH  = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv'
BASE_EXT   = '/kaggle/input/datasets/debashish311601/formula-1-official-data-19502022/'
OUT        = '/kaggle/working/'
SEED, FOLDS = 42, 5

NB_ROOTS = [
    '/kaggle/input/notebooks/mohamedsadokaissaoui/ps6e5-ensemble-0-95314-best-score',
    '/kaggle/input/notebooks/yekenot/ps-s6-e5-realmlp-pytabkit',
    '/kaggle/input/notebooks/analyticaobscura/pit-or-stay-f1-strategy-1',
    '/kaggle/input/notebooks/anthonytherrien/predicting-f1-pit-stops-blend',
    '/kaggle/input/notebooks/huzaifa242/predicting-f1-pit-stops-ensemble',
]
SVANIK_PATH = '/kaggle/input/svanik-dataset/submission (6).csv'

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
print(f"PyTabKit: {version('pytabkit')} | LGB: {lgb.__version__} | XGB: {xgb.__version__}")


In [ ]:
%%time
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

try:
    orig = pd.read_csv(ORIG_PATH)
    if 'Normalized_TyreLife' in orig.columns:
        orig = orig.drop('Normalized_TyreLife', axis=1)
    HAS_ORIG = True
    print(f"Original dataset: {orig.shape}")
except Exception as e:
    orig = None; HAS_ORIG = False
    print(f"No original dataset ({e})")

TEST_IDS = test['id'].values.copy()
print(f"Train: {train.shape} | Test: {test.shape}")
print("Target dist:", train['PitNextLap'].value_counts(normalize=True).round(3).to_dict())


In [ ]:
def _ndrv(s):
    return str(s).strip().split()[-1].lower()

def _nrace(s):
    s = str(s).strip().lower()
    s = re.sub(r'grand\s+prix|\bgp\b', '', s).strip()
    return s

def _find_col(df, *keywords, exclude=()):
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in keywords) and not any(e in cl for e in exclude):
            return c
    return None

def _load_clean(path, label):
    try:
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()
        print(f"  ✅ {label}: {df.shape[0]:,} rows")
        return df
    except Exception as e:
        print(f"  ❌ {label}: {e}")
        return None

print("Loading F1 Official Data 1950-2022...")
HAS_EXT = False; HAS_HIST = False; HIST_PIT_PRIORS = {}; EXT_DF = pd.DataFrame()

_hp = _load_clean(BASE_EXT + "pitstops.csv", "pitstops")
if _hp is not None:
    _dc = _find_col(_hp, 'driver'); _rc = _find_col(_hp, 'grand') or _find_col(_hp, 'race')
    _lc = _find_col(_hp, 'lap', exclude=('time',))
    if _dc and _lc:
        _hp['_dk'] = _hp[_dc].map(_ndrv); _hp['_lap'] = pd.to_numeric(_hp[_lc], errors='coerce')
        if _rc: _hp['_rk'] = _hp[_rc].map(_nrace)
        _pit_drv = _hp.dropna(subset=['_lap']).groupby('_dk').agg(
            pit_hist_avg_lap=('_lap','mean'), pit_hist_std_lap=('_lap','std')).reset_index()
        _pit_drv['pit_hist_std_lap'] = _pit_drv['pit_hist_std_lap'].fillna(5.0)
        HIST_PIT_PRIORS['driver'] = _pit_drv.set_index('_dk').to_dict(orient='index')
        if _rc:
            _pit_ckt = _hp.dropna(subset=['_lap']).groupby('_rk').agg(
                pit_ckt_avg_lap=('_lap','mean'), pit_ckt_std_lap=('_lap','std')).reset_index()
            _pit_ckt['pit_ckt_std_lap'] = _pit_ckt['pit_ckt_std_lap'].fillna(8.0)
            HIST_PIT_PRIORS['circuit'] = _pit_ckt.set_index('_rk').to_dict(orient='index')
        HAS_HIST = True
        print(f"  pitstop priors: {len(HIST_PIT_PRIORS.get('driver',{}))} drivers, {len(HIST_PIT_PRIORS.get('circuit',{}))} circuits")

_sg = _load_clean(BASE_EXT + "starting_grids.csv", "grids")
_sg_df = None
if _sg is not None:
    _dc = _find_col(_sg,'driver'); _rc = _find_col(_sg,'grand') or _find_col(_sg,'race')
    _yc = _find_col(_sg,'season') or _find_col(_sg,'year'); _gc = _find_col(_sg,'grid') or _find_col(_sg,'pos',exclude=('disq','ret'))
    _tc = _find_col(_sg,'car') or _find_col(_sg,'team')
    if _dc and _gc:
        _sg['_dk'] = _sg[_dc].map(_ndrv); _sg['_rk'] = _sg[_rc].map(_nrace) if _rc else 'unknown'
        _sg['_yr'] = pd.to_numeric(_sg[_yc], errors='coerce').fillna(0).astype(int) if _yc else 0
        _sg['_grid'] = pd.to_numeric(_sg[_gc].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
        _sg['_team'] = _sg[_tc].map(_ndrv) if _tc else 'unknown'
        _sg_df = _sg[['_dk','_rk','_yr','_grid','_team']].dropna(subset=['_grid']).copy()
        _sg_df['_grid'] = _sg_df['_grid'].astype(int)

_qu = _load_clean(BASE_EXT + "qualifyings.csv", "qualifyings"); _qu_df = None
if _qu is not None:
    _dc = _find_col(_qu,'driver'); _rc = _find_col(_qu,'grand') or _find_col(_qu,'race')
    _yc = _find_col(_qu,'season') or _find_col(_qu,'year'); _pc = _find_col(_qu,'pos',exclude=('disq',))
    if _dc and _pc:
        _qu['_dk'] = _qu[_dc].map(_ndrv); _qu['_rk'] = _qu[_rc].map(_nrace) if _rc else 'unknown'
        _qu['_yr'] = pd.to_numeric(_qu[_yc], errors='coerce').fillna(0).astype(int) if _yc else 0
        _qu['_quali'] = pd.to_numeric(_qu[_pc].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
        _qu_df = _qu[['_dk','_rk','_yr','_quali']].dropna(subset=['_quali']).copy()
        _qu_df['_quali'] = _qu_df['_quali'].astype(int)

_ds = _load_clean(BASE_EXT + "driver_standings.csv", "driver_standings"); _ds_df = None
if _ds is not None:
    _dc = _find_col(_ds,'driver'); _yc = _find_col(_ds,'season') or _find_col(_ds,'year')
    _pc = _find_col(_ds,'pos',exclude=('disq',)); _ptc = _find_col(_ds,'point')
    if _dc and _pc:
        _ds['_dk'] = _ds[_dc].map(_ndrv); _ds['_yr'] = pd.to_numeric(_ds[_yc], errors='coerce').fillna(0).astype(int) if _yc else 0
        _ds['_rk'] = 'unknown'
        _ds['_dpos'] = pd.to_numeric(_ds[_pc].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
        _ds['_dpts'] = pd.to_numeric(_ds[_ptc], errors='coerce') if _ptc else 0.0
        _ds_df = _ds[['_dk','_rk','_yr','_dpos','_dpts']].dropna(subset=['_dpos']).copy()

_cs = _load_clean(BASE_EXT + "constructor_standings.csv", "constructor_standings"); _cs_df = None
if _cs is not None:
    _tc = _find_col(_cs,'car') or _find_col(_cs,'team'); _yc = _find_col(_cs,'season') or _find_col(_cs,'year')
    _pc = _find_col(_cs,'pos',exclude=('disq',)); _ptc = _find_col(_cs,'point')
    if _tc and _pc:
        _cs['_tk'] = _cs[_tc].map(_ndrv); _cs['_yr'] = pd.to_numeric(_cs[_yc], errors='coerce').fillna(0).astype(int) if _yc else 0
        _cs['_rk'] = 'unknown'
        _cs['_cpos'] = pd.to_numeric(_cs[_pc].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
        _cs['_cpts'] = pd.to_numeric(_cs[_ptc], errors='coerce') if _ptc else 0.0
        _cs_df = _cs[['_tk','_rk','_yr','_cpos','_cpts']].dropna(subset=['_cpos']).copy()

_base_keys = None
for _df in [_sg_df, _qu_df, _ds_df]:
    if _df is not None: _base_keys = _df[['_dk','_rk','_yr']].drop_duplicates(); break
if _base_keys is not None:
    EXT_DF = _base_keys.copy()
    if _sg_df is not None: EXT_DF = EXT_DF.merge(_sg_df, on=['_dk','_rk','_yr'], how='left')
    if _qu_df is not None: EXT_DF = EXT_DF.merge(_qu_df, on=['_dk','_rk','_yr'], how='left')
    if _ds_df is not None: EXT_DF = EXT_DF.merge(_ds_df, on=['_dk','_rk','_yr'], how='left')
    if _cs_df is not None and '_team' in EXT_DF.columns:
        EXT_DF = EXT_DF.merge(_cs_df.rename(columns={'_tk':'_team'}), on=['_team','_rk','_yr'], how='left')
    HAS_EXT = True
    print(f"EXT_DF ready: {len(EXT_DF):,} rows")
else:
    print("No external lookup loaded")


---
## Pipeline A — Engineered Features (LGB · XGB · CatBoost · RealMLP-A)

**v8 addition inside `make_features_A`**: compound-normalized TyreLife + pit_imminent flag.
The per-driver-race CV target encoding is done separately after this cell (see Section: CV Target Encoding).


In [ ]:
%%time
FS_A = {}
COMBO_COLS    = [('Race','Compound'), ('Race','Year'), ('Driver','Compound')]
COMBO_NAMES_A = [f"{c1}_{c2}_" for c1, c2 in COMBO_COLS]

COMPOUND_MAX_LIFE_MAP = {'SOFT': 15, 'MEDIUM': 30, 'HARD': 50, 'INTERMEDIATE': 25, 'WET': 20}

def make_features_A(df, fit=False):
    df = df.copy().sort_values(['Driver','Race','Year','LapNumber']).reset_index(drop=True)
    g       = df.groupby(['Driver','Race','Year'])
    g_stint = df.groupby(['Driver','Race','Year','Stint'])

    df['tyre_life_sq']        = df['TyreLife'] ** 2
    df['tyre_life_log']       = np.log1p(df['TyreLife'])
    df['tyre_life_sqrt']      = np.sqrt(df['TyreLife'])
    df['deg_per_lap']         = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
    df['compound_life_ratio'] = df['TyreLife'] / (df.groupby('Compound')['TyreLife'].transform('max') + 1e-9)
    df['compound_max_life']   = df['Compound'].map(COMPOUND_MAX_LIFE_MAP).fillna(30)
    df['compound_tyre_norm']  = (df['TyreLife'] / df['compound_max_life']).clip(0, 2)
    df['tyre_overdue_norm']   = (df['compound_tyre_norm'] > 0.85).astype(int)

    df['est_total_laps']     = (df['LapNumber'] / (df['RaceProgress'] + 1e-9)).round().clip(30, 80)
    df['laps_remaining']     = (df['est_total_laps'] - df['LapNumber']).clip(lower=0)
    df['tyre_pct_remaining'] = df['TyreLife'] / (df['laps_remaining'] + 1)
    df['is_pit_window']      = ((df['RaceProgress'] >= 0.28) & (df['RaceProgress'] <= 0.62)).astype(int)
    df['is_late_race']       = (df['RaceProgress'] > 0.75).astype(int)
    df['position_pressure']  = df['Position'] * (1 - df['RaceProgress'])
    df['urgency_score']      = df['Cumulative_Degradation'].abs() * (1 - df['RaceProgress'])
    df['race_phase']         = pd.cut(df['RaceProgress'], bins=[0,.25,.5,.75,1.01], labels=[0,1,2,3]).astype(float)
    df['norm_position']      = 1 - (df['Position'] - 1) / 19.0
    df['life_x_progress']    = df['TyreLife'] * df['RaceProgress']

    df['delta_lag1']   = g['LapTime_Delta'].shift(1)
    df['delta_lag2']   = g['LapTime_Delta'].shift(2)
    df['prev_pit']     = g['PitStop'].shift(1).fillna(0)
    df['delta_accel']  = df['LapTime_Delta'] - df['delta_lag1']
    df['roll3_lt']     = g['LapTime (s)'].transform(lambda x: x.rolling(3,  min_periods=1).mean())
    df['roll5_lt']     = g['LapTime (s)'].transform(lambda x: x.rolling(5,  min_periods=1).mean())
    df['roll7_lt']     = g['LapTime (s)'].transform(lambda x: x.rolling(7,  min_periods=1).mean())
    df['roll10_lt']    = g['LapTime (s)'].transform(lambda x: x.rolling(10, min_periods=1).mean())
    df['roll15_lt']    = g['LapTime (s)'].transform(lambda x: x.rolling(15, min_periods=1).mean())
    df['roll3_d']      = g['LapTime_Delta'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['roll7_d']      = g['LapTime_Delta'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['roll3_std']    = g['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
    df['lap_vs_r3']    = df['LapTime (s)'] - df['roll3_lt']
    df['lap_vs_r5']    = df['LapTime (s)'] - df['roll5_lt']
    df['lap_vs_r7']    = df['LapTime (s)'] - df['roll7_lt']
    df['lap_vs_r10']   = df['LapTime (s)'] - df['roll10_lt']
    df['deg_velocity'] = g['Cumulative_Degradation'].transform(lambda x: x.diff(3).fillna(0) / 3)
    df['is_slow_lap']  = (df['LapTime (s)'] > df['roll5_lt'] * 1.15).astype(int)
    df['lap_in_stint']    = g_stint.cumcount()
    df['stint_start_lap'] = g_stint['LapNumber'].transform('min')

    if fit:
        FS_A['pit_laps']    = df[df['PitNextLap']==1].groupby(['Race','Year'])['LapNumber'].mean().rename('race_avg_pit_lap')
        FS_A['total_laps']  = df.groupby(['Race','Year'])['LapNumber'].max().rename('race_total_laps')
        FS_A['comp_life']   = df[df['PitNextLap']==1].groupby('Compound')['TyreLife'].mean().rename('compound_avg_life')
        FS_A['race_stints'] = df.groupby(['Race','Year'])['Stint'].max().rename('race_max_stint')
    df = df.merge(FS_A['pit_laps'].reset_index(),    on=['Race','Year'], how='left')
    df = df.merge(FS_A['total_laps'].reset_index(),  on=['Race','Year'], how='left')
    df = df.merge(FS_A['comp_life'].reset_index(),   on='Compound',      how='left')
    df = df.merge(FS_A['race_stints'].reset_index(), on=['Race','Year'], how='left')
    df['pit_window_flag']     = (np.abs(df['LapNumber'] - df['race_avg_pit_lap'].fillna(35)) <= 3).astype(int)
    df['tyre_vs_comp_avg']    = df['TyreLife'] - df['compound_avg_life'].fillna(25)
    df['overdue_pit']         = (df['TyreLife'] > df['compound_avg_life'].fillna(25)).astype(int)
    df['laps_remaining_race'] = df['race_total_laps'].fillna(60) - df['LapNumber']
    df['tyre_age_pct_race']   = df['TyreLife'] / (df['race_total_laps'].fillna(60) + 1)
    df['stint_progress']      = df['Stint'] / (df['race_max_stint'].fillna(3) + 1)
    df['tyre_life_pct']       = df['TyreLife'] / df['compound_avg_life'].fillna(25).clip(lower=1)
    df['stint_end_est']       = df['stint_start_lap'] + df['compound_avg_life'].fillna(25)
    df['laps_until_stop']     = (df['stint_end_est'] - df['LapNumber']).clip(lower=-20)
    df['pit_imminent']        = (df['laps_until_stop'] <= 2).astype(int)
    df['pit_in_5']            = (df['laps_until_stop'] <= 5).astype(int)

    df['cliff_flag']       = ((df['LapTime_Delta'] > df['roll3_d'].fillna(0) + 2*df['roll3_std'].fillna(0)) & (df['TyreLife'] > 15)).astype(int)
    df['laps_to_stop']     = df['compound_avg_life'].fillna(25) - df['TyreLife']
    df['past_optimal']     = (df['laps_to_stop'] < -3).astype(int)
    df['must_pit_or_stay'] = (df['laps_remaining_race'] < df['TyreLife'] * 0.4).astype(int)
    df['undercut_threat']  = ((df['Position'] <= 10) & (df['is_pit_window'] == 1)).astype(int)
    df['relative_stint']   = df['Stint'] / df['race_max_stint'].fillna(3).clip(lower=1)
    df['deg_x_win']        = df['Cumulative_Degradation'] * df['is_pit_window']
    df['over_x_win']       = df['overdue_pit'] * df['is_pit_window']
    df['tyre_x_pres']      = df['TyreLife'] * df['position_pressure']
    df['r3_x_life']        = df['roll3_d'].fillna(0) * df['TyreLife']
    df['comp_stint']       = df['Compound'].map({'SOFT':2,'MEDIUM':1,'HARD':0}).fillna(1) * df['Stint']
    df['lap_div_rp']       = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['tl_div_ln']        = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['lap_accel_smooth'] = g['LapTime_Delta'].transform(lambda x: x.rolling(5, min_periods=2).mean().diff().fillna(0))
    df['tyre_cliff_imminent'] = ((df['lap_accel_smooth'] > 0.3) & (df['TyreLife'] > 12)).astype(int)
    df['laps_to_window_start'] = (df['race_avg_pit_lap'].fillna(35) * 0.85 - df['LapNumber']).clip(-20, 30)
    df['laps_to_window_end']   = (df['race_avg_pit_lap'].fillna(35) * 1.15 - df['LapNumber']).clip(-20, 30)
    df['in_optimal_window']    = ((df['laps_to_window_start'] <= 0) & (df['laps_to_window_end'] >= 0)).astype(int)
    df['stint_lt_baseline']    = g_stint['LapTime (s)'].transform('first')
    df['stint_degradation_ratio'] = ((df['LapTime (s)'] - df['stint_lt_baseline']) / (df['stint_lt_baseline'] + 1e-9)).clip(-0.1, 0.3)
    if fit:
        FS_A['compound_race_lt'] = df.groupby(['Race','Year','Compound'])['LapTime (s)'].median().rename('compound_race_median_lt')
    df = df.merge(FS_A['compound_race_lt'].reset_index(), on=['Race','Year','Compound'], how='left')
    df['lap_vs_compound_baseline'] = (df['LapTime (s)'] - df['compound_race_median_lt'].fillna(df['LapTime (s)'])).clip(-5, 15)
    df['roll7_std']    = g['LapTime (s)'].transform(lambda x: x.rolling(7, min_periods=2).std().fillna(0))
    df['roll3_var_ratio'] = (df['roll3_std'] / (df['roll7_std'] + 1e-9)).clip(0, 5)
    df['pos_lag3']     = g['Position'].shift(3)
    df['pos_trend_3']  = (df['Position'] - df['pos_lag3'].fillna(df['Position'])).clip(-10, 10)
    df['losing_positions'] = (df['pos_trend_3'] > 1).astype(int)
    if fit:
        FS_A['race_compound_max'] = df.groupby(['Race','Year','Compound'])['TyreLife'].max().rename('race_compound_max_life')
    df = df.merge(FS_A['race_compound_max'].reset_index(), on=['Race','Year','Compound'], how='left')
    df['tyre_freshness_pct']  = (1 - df['TyreLife'] / (df['race_compound_max_life'].fillna(40) + 1)).clip(0, 1)
    df['must_pit_signal']     = (df['tyre_cliff_imminent'] * df['overdue_pit'] * df['in_optimal_window']).astype(int)
    df['urgency_composite']   = df['urgency_score'] * df['stint_degradation_ratio'].clip(0, 1) * (1 + df['tyre_cliff_imminent'])
    if fit:
        FS_A['driver_compound_avg_life'] = df[df['PitNextLap']==1].groupby(['Driver','Compound'])['TyreLife'].mean().rename('dc_avg_stint_life')
    _dc_life = FS_A.get('driver_compound_avg_life')
    if _dc_life is not None:
        df = df.merge(_dc_life.reset_index(), on=['Driver','Compound'], how='left')
    else:
        df['dc_avg_stint_life'] = 25.0
    df['driver_vs_avg_life']      = (df['TyreLife'] - df['dc_avg_stint_life'].fillna(25)).clip(-20, 20)
    df['driver_overdue_personal'] = (df['TyreLife'] > df['dc_avg_stint_life'].fillna(25)).astype(int)

    df['compound_ord'] = df['Compound'].map({'SOFT':2,'MEDIUM':1,'HARD':0,'INTERMEDIATE':3,'WET':4}).fillna(1)
    df['year_le']      = df['Year'] - df['Year'].min()
    base_nums = ['PitStop','LapNumber','Stint','TyreLife','Position','LapTime (s)','LapTime_Delta','Cumulative_Degradation','RaceProgress','Position_Change']
    for col in base_nums:
        cname = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            FS_A[f'nc_{col}'] = {v: i for i, v in enumerate(uniques)}
        df[cname] = np.floor(df[col]).map(FS_A[f'nc_{col}']).fillna(-1).astype('int32').astype(str)
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal', strategy='quantile', subsample=None)
        df['rp_bin_'] = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
        FS_A['kb'] = kb
    else:
        df['rp_bin_'] = FS_A['kb'].transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
    for c1, c2 in COMBO_COLS:
        cn = f"{c1}_{c2}_"
        combo = df[c1].astype(str) + '_' + df[c2].astype(str)
        if fit:
            codes, uniques = combo.factorize()
            FS_A[cn] = {v: i for i, v in enumerate(uniques)}
        df[cn] = combo.map(FS_A[cn]).fillna(-1).astype('int32').astype(str)

    from sklearn.model_selection import StratifiedKFold as _SKF
    if fit and 'PitNextLap' in df.columns:
        teskf = _SKF(5, shuffle=True, random_state=SEED)
        gm = df['PitNextLap'].mean()
        for col in ['Driver','Race']:
            df[f'{col}_te'] = 0.0
            for ti, vi in teskf.split(df, df['PitNextLap']):
                m = df.iloc[ti].groupby(col)['PitNextLap'].mean()
                df.iloc[vi, df.columns.get_loc(f'{col}_te')] = df.iloc[vi][col].map(m).fillna(gm).values
            FS_A[f'{col}_te_map'] = df.groupby(col)['PitNextLap'].mean().to_dict()
        for (c1, c2), te_name in [(('Driver','Compound'), 'Driver_Compound_te'), (('Race','Compound'), 'Race_Compound_te')]:
            combined = (df[c1].astype(str) + '_' + df[c2].astype(str)).values
            df[te_name] = gm
            for ti, vi in teskf.split(df, df['PitNextLap']):
                m = pd.Series(df['PitNextLap'].values[ti], index=combined[ti]).groupby(level=0).mean()
                df.iloc[vi, df.columns.get_loc(te_name)] = pd.Series(combined[vi]).map(m).fillna(gm).values
            FS_A[f'{te_name}_map'] = pd.Series(df['PitNextLap'].values, index=combined).groupby(level=0).mean().to_dict()
    else:
        for col in ['Driver','Race']:
            gm = float(np.mean(list(FS_A.get(f'{col}_te_map', {}).values()) or [0.2]))
            df[f'{col}_te'] = df[col].map(FS_A.get(f'{col}_te_map', {})).fillna(gm)
        for (c1, c2), te_name in [(('Driver','Compound'), 'Driver_Compound_te'), (('Race','Compound'), 'Race_Compound_te')]:
            combined = (df[c1].astype(str) + '_' + df[c2].astype(str)).values
            gm = float(np.mean(list(FS_A.get(f'{te_name}_map', {}).values()) or [0.2]))
            df[te_name] = pd.Series(combined).map(FS_A.get(f'{te_name}_map', {})).fillna(gm).values

    df['_dk'] = df['Driver'].map(_ndrv); df['_rk'] = df['Race'].map(_nrace); df['_yr'] = df['Year'].astype(int)
    if HAS_HIST:
        drv_p = HIST_PIT_PRIORS.get('driver', {}); ckt_p = HIST_PIT_PRIORS.get('circuit', {})
        df['pit_hist_avg_lap']  = df['_dk'].map(lambda k: drv_p.get(k,{}).get('pit_hist_avg_lap', 30.0))
        df['pit_hist_std_lap']  = df['_dk'].map(lambda k: drv_p.get(k,{}).get('pit_hist_std_lap',  8.0))
        df['drv_laps_vs_hist']  = (df['TyreLife'] - df['pit_hist_avg_lap']).clip(-25, 25)
        df['drv_hist_overdue']  = (df['TyreLife'] > df['pit_hist_avg_lap']).astype(int)
        df['ckt_hist_avg_lap']  = df['_rk'].map(lambda k: ckt_p.get(k,{}).get('pit_ckt_avg_lap', 28.0))
        df['ckt_hist_std_lap']  = df['_rk'].map(lambda k: ckt_p.get(k,{}).get('pit_ckt_std_lap',  8.0))
        df['laps_vs_ckt_hist']  = (df['TyreLife'] - df['ckt_hist_avg_lap']).clip(-25, 25)
        df['in_ckt_pit_window'] = (df['laps_vs_ckt_hist'].abs() <= df['ckt_hist_std_lap']).astype(int)
    else:
        for _c in ['pit_hist_avg_lap','pit_hist_std_lap','drv_laps_vs_hist','drv_hist_overdue','ckt_hist_avg_lap','ckt_hist_std_lap','laps_vs_ckt_hist','in_ckt_pit_window']:
            df[_c] = 0.0
    if HAS_EXT and len(EXT_DF) > 0:
        _merged = df[['_dk','_rk','_yr']].merge(EXT_DF, on=['_dk','_rk','_yr'], how='left')
        df['grid_position']  = _merged['_grid'].fillna(10).values.astype(float) if '_grid' in _merged else 10.0
        df['grid_vs_pos']    = (df['Position'] - df['grid_position']).clip(-15, 15)
        df['started_front5'] = (df['grid_position'] <= 5).astype(int)
        df['quali_position'] = _merged['_quali'].fillna(10).values.astype(float) if '_quali' in _merged else 10.0
        df['drv_champ_pos']  = _merged['_dpos'].fillna(10).values.astype(float) if '_dpos' in _merged else 10.0
        df['drv_champ_pts']  = _merged['_dpts'].fillna(0).values.astype(float)  if '_dpts' in _merged else 0.0
        df['cons_champ_pos'] = _merged['_cpos'].fillna(5).values.astype(float)  if '_cpos' in _merged else 5.0
    else:
        for _c in ['grid_position','grid_vs_pos','started_front5','quali_position','drv_champ_pos','drv_champ_pts','cons_champ_pos']:
            df[_c] = 0.0
    df.drop(columns=['_dk','_rk','_yr'], inplace=True, errors='ignore')
    return df.fillna(0)

print("Engineering train A...")
train_A = make_features_A(train, fit=True)
print("Engineering test A...")
test_A  = make_features_A(test,  fit=False)
if HAS_ORIG:
    print("Engineering original A...")
    orig_A = make_features_A(orig, fit=False)
else:
    orig_A = None

CAT_COLS_A = [c for c in train_A.columns if c.endswith('_')]
DROP_A     = ['id','Driver','Compound','Race','Year','PitNextLap'] + CAT_COLS_A
TREE_FEATS = [c for c in train_A.columns if c not in DROP_A]
MLP_FEATS_A= TREE_FEATS + CAT_COLS_A

X      = train_A[TREE_FEATS].astype(np.float32)
y      = train_A['PitNextLap'].astype(int)
X_test = test_A[TREE_FEATS].astype(np.float32)

if orig_A is not None:
    X_orig = orig_A.reindex(columns=TREE_FEATS, fill_value=0).astype(np.float32)
    y_orig = orig_A['PitNextLap'].astype(int)
else:
    X_orig = y_orig = None

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

def prep_mlp(df, feats, cat_cols):
    d = df.reindex(columns=feats, fill_value=0).copy()
    for c in cat_cols:
        if c in d.columns: d[c] = d[c].astype('category')
    return d

X_mlp_A      = prep_mlp(train_A, MLP_FEATS_A, CAT_COLS_A)
X_test_mlp_A = prep_mlp(test_A,  MLP_FEATS_A, CAT_COLS_A)
X_orig_mlp_A = prep_mlp(orig_A,  MLP_FEATS_A, CAT_COLS_A) if orig_A is not None else None
print(f"Tree feats: {len(TREE_FEATS)} | MLP-A feats: {len(MLP_FEATS_A)}")


---
## Pipeline B — Raw Features (RealMLP-B)

In [ ]:
%%time
FS_B = {}
COMBO_NAMES_B = [f"{c1}_{c2}_b_" for c1, c2 in COMBO_COLS]

def make_features_B(df, fit=False):
    df = df.copy().sort_values(['Driver','Race','Year','LapNumber']).reset_index(drop=True)
    base_num = ['PitStop','LapNumber','Stint','TyreLife','Position','LapTime (s)','LapTime_Delta','Cumulative_Degradation','RaceProgress','Position_Change']
    for col in base_num:
        cn = f"{col}_bcat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            FS_B[f'nc_{col}'] = {v: i for i, v in enumerate(uniques)}
        df[cn] = np.floor(df[col]).map(FS_B[f'nc_{col}']).fillna(-1).astype('int32').astype(str)
    df['_b_lap_div_rp'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_b_tl_div_ln']  = (df['TyreLife']  / df['LapNumber'].clip(lower=1)).astype('float32')
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal', strategy='quantile', subsample=None)
        df['rp200_b_'] = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
        FS_B['kb'] = kb
    else:
        df['rp200_b_'] = FS_B['kb'].transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
    for (c1, c2), cn in zip(COMBO_COLS, COMBO_NAMES_B):
        combo = df[c1].astype(str) + '_' + df[c2].astype(str)
        if fit:
            codes, uniques = combo.factorize()
            FS_B[cn] = {v: i for i, v in enumerate(uniques)}
        df[cn] = combo.map(FS_B[cn]).fillna(-1).astype('int32').astype(str)
    return df

print("Engineering train B..."); train_B = make_features_B(train, fit=True)
print("Engineering test B...");  test_B  = make_features_B(test,  fit=False)
orig_B = make_features_B(orig, fit=False) if HAS_ORIG else None

CAT_COLS_B  = [c for c in train_B.columns if c.endswith('_b_') or c == 'rp200_b_' or c.endswith('_bcat_')]
NUM_COLS_B  = [c for c in train_B.columns if c.startswith('_b_')]
RAW_NUM_B   = ['PitStop','LapNumber','Stint','TyreLife','Position','LapTime (s)','LapTime_Delta','Cumulative_Degradation','RaceProgress','Position_Change']
MLP_FEATS_B = RAW_NUM_B + NUM_COLS_B + CAT_COLS_B

def prep_mlp_b(df, feats, cat_cols):
    d = df.reindex(columns=feats, fill_value=0).copy()
    for c in cat_cols:
        if c in d.columns: d[c] = d[c].astype('category')
    return d

X_mlp_B      = prep_mlp_b(train_B, MLP_FEATS_B, CAT_COLS_B)
X_test_mlp_B = prep_mlp_b(test_B,  MLP_FEATS_B, CAT_COLS_B)
X_orig_mlp_B = prep_mlp_b(orig_B,  MLP_FEATS_B, CAT_COLS_B) if orig_B is not None else None
print(f"MLP-B feats: {len(MLP_FEATS_B)}")


## 🎯 Target Encoding: Driver × Race × Year

Among all feature engineering decisions in this pipeline, 
**CV target encoding** of high-cardinality combinations gave 
the single biggest lift — approximately **+0.002 OOF AUC** 
over frequency encoding alone.

### Why these three combinations?

A pit stop decision is not just about tyre wear — it's about 
*who* is driving, *where* they are racing, and *when* in F1 
history. Three encoded combinations capture this:

| Feature | What it encodes | Why it matters |
|---------|----------------|----------------|
| `Driver × Race` | This driver's historical pit behavior at this circuit | Vettel at Monaco pits differently than Hamilton at Monza |
| `Driver × Race × Year` | Same, but year-specific | Teams change strategy year over year — 2021 Red Bull ≠ 2019 Red Bull |
| `Driver × Compound` | How aggressively this driver uses each tyre type | Some drivers are easy on softs; others push to the cliff |

### Why target encoding beats ordinal/frequency here

Ordinal encoding treats `Hamilton__Monaco` and 
`Latifi__Monaco` as arbitrary integers — it conveys no 
information about pit likelihood. Frequency encoding only 
tells you *how often* a combination appears, not *what 
happens* when it does.

Target encoding replaces the category with the **mean pit 
rate** for that group, smoothed to avoid overfitting rare 
combinations:

TE(cat) = (count × mean_pit_rate + smoothing × global_mean) : (count + smoothing)


For `Hamilton__Bahrain__2021`: if Hamilton pitted on 18% of 
laps at Bahrain in 2021, the encoded value is ~0.18 — a 
direct probability estimate the model can act on immediately.

### Leak prevention

The critical detail: encoding is computed **inside each fold** 
on the training portion only, then applied to validation. 
This prevents the model from seeing target information from 
the rows it's being evaluated on — a common mistake that 
inflates OOF scores without improving LB.

In [ ]:
def cv_target_encode(train_df, test_df, group_cols, target, folds, smoothing=30):
    global_mean = target.mean()
    oof_enc  = np.full(len(train_df), global_mean, dtype=np.float32)
    test_enc = np.zeros(len(test_df),  dtype=np.float32)
    key      = train_df[group_cols].astype(str).agg('__'.join, axis=1)
    key_test = test_df[group_cols].astype(str).agg('__'.join, axis=1)
    for ti, vi in folds:
        stats = (pd.DataFrame({'key': key.iloc[ti], 'target': target.iloc[ti]})
                   .groupby('key')['target'].agg(['sum','count']))
        stats['enc'] = (stats['sum'] + smoothing * global_mean) / (stats['count'] + smoothing)
        oof_enc[vi] = key.iloc[vi].map(stats['enc'].to_dict()).fillna(global_mean).values
    stats_full = (pd.DataFrame({'key': key, 'target': target})
                    .groupby('key')['target'].agg(['sum','count']))
    stats_full['enc'] = (stats_full['sum'] + smoothing * global_mean) / (stats_full['count'] + smoothing)
    test_enc = key_test.map(stats_full['enc'].to_dict()).fillna(global_mean).values
    return oof_enc.astype(np.float32), test_enc.astype(np.float32)

fold_list = list(skf.split(train_A, y))

te_configs = [
    (['Driver','Race','Year'],        20,  'te_drv_race_yr'),
    (['Driver','Race'],               30,  'te_drv_race'),
    (['Race','Compound'],             25,  'te_race_comp'),
    (['Driver','Compound'],           25,  'te_drv_comp'),
    (['Race','Year'],                 20,  'te_race_yr'),
    (['Driver','Race','Compound'],    15,  'te_drv_race_comp'),
]

te_train_dict = {}; te_test_dict = {}
for cols, smooth, name in te_configs:
    if all(c in train_A.columns for c in cols):
        oof_enc, tst_enc = cv_target_encode(train_A, test_A, cols, y, fold_list, smoothing=smooth)
        te_train_dict[name] = oof_enc; te_test_dict[name] = tst_enc
        print(f"  {name:25s} | OOF mean={oof_enc.mean():.4f} | std={oof_enc.std():.4f}")
    else:
        print(f"  {name}: missing cols")
print(f"Encoded {len(te_train_dict)} TE features")


---
## Model 1 — LightGBM (GPU) ~9 min

In [ ]:
%%time
warnings.filterwarnings('ignore')
LGB_DEVICE = 'gpu' if DEVICE == 'cuda' else 'cpu'

lgb_params = dict(
    objective='binary', metric='auc', n_estimators=6000, learning_rate=0.025,
    num_leaves=255, min_child_samples=25, min_data_in_bin=10,
    feature_fraction=0.65, bagging_fraction=0.85, bagging_freq=1,
    lambda_l1=1.2, lambda_l2=2.5, max_depth=10, path_smooth=0.1,
    device=LGB_DEVICE, random_state=SEED, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(X)); pred_lgb = np.zeros(len(X_test)); lgb_models = []
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    Xtr, Xva, ytr, yva = X.iloc[ti], X.iloc[vi], y.iloc[ti], y.iloc[vi]
    if X_orig is not None:
        Xtr = pd.concat([Xtr, X_orig], ignore_index=True); ytr = pd.concat([ytr, y_orig], ignore_index=True)
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(1000)])
    oof_lgb[vi] = m.predict_proba(Xva)[:, 1]; pred_lgb += m.predict_proba(X_test)[:, 1] / FOLDS
    lgb_models.append(m)
    print(f"  Fold {fold+1}: {roc_auc_score(yva, oof_lgb[vi]):.5f}")

lgb_auc = roc_auc_score(y, oof_lgb)
print(f"\n{Fore.GREEN}{Style.BRIGHT}LGB OOF: {lgb_auc:.5f}{Style.RESET_ALL}")


---
## XGBoost Optuna Tuning ~4 min
Single validation fold (fold 0), 50% subsample. 4-minute timeout.

In [ ]:
%%time
_splits = list(skf.split(X, y)); ti0, vi0 = _splits[0]
rng = np.random.RandomState(SEED)
sub_idx = rng.choice(ti0, size=int(len(ti0)*0.50), replace=False)
X_tr0, y_tr0 = X.iloc[sub_idx], y.iloc[sub_idx]
X_va0, y_va0 = X.iloc[vi0],     y.iloc[vi0]
_xgb_device  = 'cuda' if DEVICE == 'cuda' else 'cpu'

def xgb_objective(trial):
    gp = trial.suggest_categorical('grow_policy', ['depthwise','lossguide'])
    params = dict(
        n_estimators=1500, learning_rate=trial.suggest_float('lr', 0.01, 0.15, log=True),
        max_depth=trial.suggest_int('max_depth', 5, 10),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.4, 0.9),
        colsample_bylevel=trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        gamma=trial.suggest_float('gamma', 0.0, 2.0),
        reg_alpha=trial.suggest_float('reg_alpha', 0.05, 5.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 0.05, 5.0, log=True),
        grow_policy=gp, tree_method='hist', device=_xgb_device,
        eval_metric='auc', early_stopping_rounds=40,
        random_state=SEED, n_jobs=-1, verbosity=0,
    )
    if gp == 'lossguide': params['max_leaves'] = trial.suggest_int('max_leaves', 64, 512)
    m = xgb.XGBClassifier(**params)
    m.fit(X_tr0, y_tr0, eval_set=[(X_va0, y_va0)], verbose=False)
    return roc_auc_score(y_va0, m.predict_proba(X_va0)[:, 1])

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective, timeout=4*60, show_progress_bar=True)
_bp = study.best_params
best_xgb_params = dict(
    n_estimators=5000, learning_rate=_bp['lr'], max_depth=_bp['max_depth'],
    min_child_weight=_bp['min_child_weight'], subsample=_bp['subsample'],
    colsample_bytree=_bp['colsample_bytree'], colsample_bylevel=_bp['colsample_bylevel'],
    gamma=_bp['gamma'], reg_alpha=_bp['reg_alpha'], reg_lambda=_bp['reg_lambda'],
    grow_policy=_bp['grow_policy'], tree_method='hist', device=_xgb_device,
    eval_metric='auc', early_stopping_rounds=150, random_state=SEED, n_jobs=-1,
)
if _bp['grow_policy'] == 'lossguide': best_xgb_params['max_leaves'] = _bp['max_leaves']
print(f"Optuna best val AUC: {study.best_value:.5f}  ({len(study.trials)} trials)")


---
## Model 2 — XGBoost (GPU, Optuna-tuned) ~3 min

In [ ]:
%%time
oof_xgb = np.zeros(len(X)); pred_xgb = np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    Xtr, Xva, ytr, yva = X.iloc[ti], X.iloc[vi], y.iloc[ti], y.iloc[vi]
    if X_orig is not None:
        Xtr = pd.concat([Xtr, X_orig], ignore_index=True); ytr = pd.concat([ytr, y_orig], ignore_index=True)
    m = xgb.XGBClassifier(**best_xgb_params)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=500)
    oof_xgb[vi] = m.predict_proba(Xva)[:, 1]; pred_xgb += m.predict_proba(X_test)[:, 1] / FOLDS
    print(f"  Fold {fold+1}: {roc_auc_score(yva, oof_xgb[vi]):.5f}")

xgb_auc = roc_auc_score(y, oof_xgb)
print(f"\n{Fore.GREEN}{Style.BRIGHT}XGB OOF: {xgb_auc:.5f}{Style.RESET_ALL}")


---
## Model 3 — CatBoost (GPU) ~3 min

In [ ]:
%%time
cb_params = dict(
    iterations=5000, learning_rate=0.05, depth=8, l2_leaf_reg=3,
    bootstrap_type='Bernoulli', subsample=0.8, eval_metric='AUC',
    task_type='GPU' if DEVICE == 'cuda' else 'CPU',
    random_seed=SEED, verbose=500, early_stopping_rounds=100,
)
oof_cb = np.zeros(len(X)); pred_cb = np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    Xtr, Xva, ytr, yva = X.iloc[ti], X.iloc[vi], y.iloc[ti], y.iloc[vi]
    if X_orig is not None:
        Xtr = pd.concat([Xtr, X_orig], ignore_index=True); ytr = pd.concat([ytr, y_orig], ignore_index=True)
    m = cb.CatBoostClassifier(**cb_params)
    m.fit(Xtr, ytr, eval_set=(Xva, yva))
    oof_cb[vi] = m.predict_proba(Xva)[:, 1]; pred_cb += m.predict_proba(X_test)[:, 1] / FOLDS
    print(f"  Fold {fold+1}: {roc_auc_score(yva, oof_cb[vi]):.5f}")

cb_auc = roc_auc_score(y, oof_cb)
print(f"\n{Fore.GREEN}{Style.BRIGHT}CatBoost OOF: {cb_auc:.5f}{Style.RESET_ALL}")


---
## Pseudo-Labeling
High-confidence test rows added to MLP training. Strict thresholds (0.97/0.015) to avoid class imbalance.

In [ ]:
pseudo_prob = (pred_lgb + pred_xgb + pred_cb) / 3.0
PSEUDO_POS_THRESH = 0.97; PSEUDO_NEG_THRESH = 0.015
pseudo_mask = (pseudo_prob > PSEUDO_POS_THRESH) | (pseudo_prob < PSEUDO_NEG_THRESH)
pseudo_idx  = np.where(pseudo_mask)[0]
y_pseudo    = pd.Series((pseudo_prob[pseudo_mask] > 0.5).astype(int), name='PitNextLap').reset_index(drop=True)
pos_rate    = (y_pseudo == 1).mean()
print(f"Pseudo rows: {pseudo_mask.sum():,} ({pseudo_mask.mean()*100:.1f}%) | positive rate: {pos_rate:.3f}")

if pos_rate < 0.03:
    print("Too imbalanced — SKIPPING pseudo-labeling")
    X_extra_mlp_A = X_orig_mlp_A if orig_A is not None else pd.DataFrame()
    X_extra_mlp_B = X_orig_mlp_B if orig_B is not None else pd.DataFrame()
    y_extra = y_orig if HAS_ORIG else pd.Series(dtype=int)
else:
    X_pseudo_mlp_A = X_test_mlp_A.iloc[pseudo_idx].reset_index(drop=True)
    X_pseudo_mlp_B = X_test_mlp_B.iloc[pseudo_idx].reset_index(drop=True)
    if orig_A is not None:
        X_extra_mlp_A = pd.concat([X_orig_mlp_A, X_pseudo_mlp_A], ignore_index=True)
        X_extra_mlp_B = pd.concat([X_orig_mlp_B, X_pseudo_mlp_B], ignore_index=True)
        y_extra = pd.concat([y_orig, y_pseudo], ignore_index=True)
    else:
        X_extra_mlp_A = X_pseudo_mlp_A; X_extra_mlp_B = X_pseudo_mlp_B; y_extra = y_pseudo
    print(f"Total extra training rows: {len(y_extra):,}")


---
## Model 4 — RealMLP-A: Engineered + pseudo-labels ~8 min

In [ ]:
%%time
MLP_PARAMS = dict(
    random_state=SEED, verbosity=1, val_metric_name='1-auc_ovr',
    n_ens=8, n_epochs=4, batch_size=256, use_early_stopping=False,
    lr=0.075, wd=0.018, sq_mom=0.98, lr_sched='cos_anneal',
    first_layer_lr_factor=0.25, embedding_size=6, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu',
    plr_lr_factor=0.1151, plr_sigma=2.33, ls_eps=0.01, ls_eps_sched='sqrt_cos',
    add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot','median_center','robust_scale','smooth_clip','embedding','l2_normalize'],
)

oof_mlp_A = np.zeros(len(X)); pred_mlp_A = np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(skf.split(X, y), 1):
    print(f"### Fold {fold}/{FOLDS}")
    Xtr = X_mlp_A.iloc[ti].copy(); ytr = y.iloc[ti]
    Xva = X_mlp_A.iloc[vi].copy(); yva = y.iloc[vi]
    Xte = X_test_mlp_A.copy()
    Xtr = pd.concat([Xtr, X_extra_mlp_A], ignore_index=True)
    ytr = pd.concat([ytr, y_extra],        ignore_index=True)
    te = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
    te_enc = te.fit_transform(Xtr[COMBO_NAMES_A], ytr)
    te_names = [f'_te_{c}' for c in COMBO_NAMES_A]
    Xtr[te_names] = te_enc; Xva[te_names] = te.transform(Xva[COMBO_NAMES_A]); Xte[te_names] = te.transform(Xte[COMBO_NAMES_A])
    m = RealMLP_TD_Classifier(**MLP_PARAMS)
    m.fit(Xtr, ytr, Xva, yva)
    oof_mlp_A[vi] = m.predict_proba(Xva)[:, 1]; pred_mlp_A += m.predict_proba(Xte)[:, 1] / FOLDS
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} AUC: {roc_auc_score(yva, oof_mlp_A[vi]):.5f}{Style.RESET_ALL}")
    torch.cuda.empty_cache(); gc.collect()

mlp_A_auc = roc_auc_score(y, oof_mlp_A)
print(f"\n{Fore.GREEN}{Style.BRIGHT}RealMLP-A OOF: {mlp_A_auc:.5f}{Style.RESET_ALL}")


---
## Model 5 — RealMLP-B: Raw + pseudo-labels ~8 min

In [ ]:
%%time
oof_mlp_B = np.zeros(len(X)); pred_mlp_B = np.zeros(len(X_test))
for fold, (ti, vi) in enumerate(skf.split(X, y), 1):
    print(f"### Fold {fold}/{FOLDS}")
    Xtr = X_mlp_B.iloc[ti].copy(); ytr = y.iloc[ti]
    Xva = X_mlp_B.iloc[vi].copy(); yva = y.iloc[vi]
    Xte = X_test_mlp_B.copy()
    Xtr = pd.concat([Xtr, X_extra_mlp_B], ignore_index=True)
    ytr = pd.concat([ytr, y_extra],        ignore_index=True)
    te = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
    te_enc = te.fit_transform(Xtr[COMBO_NAMES_B], ytr)
    te_names = [f'_te_{c}' for c in COMBO_NAMES_B]
    Xtr[te_names] = te_enc; Xva[te_names] = te.transform(Xva[COMBO_NAMES_B]); Xte[te_names] = te.transform(Xte[COMBO_NAMES_B])
    m = RealMLP_TD_Classifier(**MLP_PARAMS)
    m.fit(Xtr, ytr, Xva, yva)
    oof_mlp_B[vi] = m.predict_proba(Xva)[:, 1]; pred_mlp_B += m.predict_proba(Xte)[:, 1] / FOLDS
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} AUC: {roc_auc_score(yva, oof_mlp_B[vi]):.5f}{Style.RESET_ALL}")
    torch.cuda.empty_cache(); gc.collect()

mlp_B_auc = roc_auc_score(y, oof_mlp_B)
print(f"\n{Fore.GREEN}{Style.BRIGHT}RealMLP-B OOF: {mlp_B_auc:.5f}{Style.RESET_ALL}")

_names = ['LGB','XGB','CB','RealMLP-A','RealMLP-B']
_oofs  = [oof_lgb, oof_xgb, oof_cb, oof_mlp_A, oof_mlp_B]
_preds = [pred_lgb, pred_xgb, pred_cb, pred_mlp_A, pred_mlp_B]
print("\nIndividual OOF AUCs:")
for n, o in zip(_names, _oofs):
    print(f"  {n:14s}: {roc_auc_score(y, o):.5f}")


---
## Stacking Meta-Learner v8

Stack matrix = raw OOF probs + rank-normalized + logit-transformed + pairwise products/diffs + engineered raw cols + CV target encodings.
Pairwise interactions let the meta-learner identify samples where base models disagree.


In [ ]:
%%time

def build_stack_matrix(oofs_or_preds, extra_raw_df, te_dict, train_df_ref=None):
    n   = len(oofs_or_preds[0])
    eps = 1e-6
    parts = [np.column_stack(oofs_or_preds)]
    parts.append(np.column_stack([rankdata(p) / n for p in oofs_or_preds]))
    parts.append(np.column_stack([_logit(np.clip(p, eps, 1-eps)) for p in oofs_or_preds]))
    for i in range(len(oofs_or_preds)):
        for j in range(i+1, len(oofs_or_preds)):
            a, b = oofs_or_preds[i], oofs_or_preds[j]
            parts.append((a * b).reshape(-1, 1))
            parts.append(np.abs(a - b).reshape(-1, 1))
    raw_cols = ['TyreLife','RaceProgress','lap_in_stint','tyre_life_pct','laps_until_stop',
                'compound_tyre_norm','deg_per_lap','urgency_score','position_pressure',
                'pit_imminent','pit_in_5','compound_ord','tyre_overdue_norm']
    avail = [c for c in raw_cols if c in extra_raw_df.columns]
    parts.append(extra_raw_df[avail].fillna(0).values.astype(np.float32))
    if te_dict:
        parts.append(np.column_stack(list(te_dict.values())))
    return np.concatenate(parts, axis=1).astype(np.float32)

stack_tr = build_stack_matrix(_oofs,  train_A, te_train_dict)
stack_te = build_stack_matrix(_preds, test_A,  te_test_dict)
print(f"Stack matrix: {stack_tr.shape}")

oof_stack  = np.zeros(len(y))
pred_stack = np.zeros(len(X_test))

meta_params = dict(
    objective='binary', metric='auc', n_estimators=800,
    learning_rate=0.02, num_leaves=31, min_child_samples=50,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    lambda_l1=2.0, lambda_l2=4.0,
    device=LGB_DEVICE, random_state=SEED, n_jobs=-1, verbose=-1,
)
for fold, (ti, vi) in enumerate(skf.split(stack_tr, y)):
    m = lgb.LGBMClassifier(**meta_params)
    m.fit(stack_tr[ti], y.iloc[ti], eval_set=[(stack_tr[vi], y.iloc[vi])],
          callbacks=[lgb.early_stopping(80, verbose=False)])
    oof_stack[vi]  = m.predict_proba(stack_tr[vi])[:, 1]
    pred_stack    += m.predict_proba(stack_te)[:, 1] / FOLDS
    print(f"  Fold {fold+1}: {roc_auc_score(y.iloc[vi], oof_stack[vi]):.5f}")

stack_auc = roc_auc_score(y, oof_stack)
print(f"\n{Fore.GREEN}{Style.BRIGHT}Stack v8 OOF: {stack_auc:.5f}{Style.RESET_ALL}")


---
## Internal Ensemble — Hill Climbing + Stack

In [ ]:
%%time
def neg_auc(w):
    ww = np.abs(w); ww /= (ww.sum() + 1e-12)
    return -roc_auc_score(y, sum(wi * p for wi, p in zip(ww, _oofs)))

w0s = [
    [0.22, 0.10, 0.10, 0.28, 0.30], [0.20, 0.08, 0.10, 0.25, 0.37],
    [0.18, 0.08, 0.08, 0.30, 0.36], [0.25, 0.08, 0.10, 0.22, 0.35],
    [0.20, 0.06, 0.08, 0.28, 0.38], [0.22, 0.10, 0.12, 0.24, 0.32],
]
best_hc = None
for w0 in w0s:
    r = minimize(neg_auc, x0=w0, method='Nelder-Mead', options={'maxiter':3000,'xatol':1e-7,'fatol':1e-7})
    if best_hc is None or r.fun < best_hc.fun: best_hc = r

bw = np.abs(best_hc.x); bw /= bw.sum()
prob_oof  = sum(w*p for w,p in zip(bw, _oofs));  prob_pred = sum(w*p for w,p in zip(bw, _preds))
rank_oof  = np.average([rankdata(p)/len(p) for p in _oofs],  axis=0, weights=bw)
rank_pred = np.average([rankdata(p)/len(p) for p in _preds], axis=0, weights=bw)
eq_oof    = np.mean([rankdata(p)/len(p) for p in _oofs],  axis=0)
eq_pred   = np.mean([rankdata(p)/len(p) for p in _preds], axis=0)

blend50_oof  = 0.5*prob_oof + 0.5*oof_stack;  blend50_pred = 0.5*prob_pred + 0.5*pred_stack

v7_cands = [
    (roc_auc_score(y, prob_oof),    prob_pred,    "hill-climb prob"),
    (roc_auc_score(y, rank_oof),    rank_pred,    "hill-climb rank"),
    (roc_auc_score(y, eq_oof),      eq_pred,      "equal rank"),
    (stack_auc,                     pred_stack,   "stacking v8"),
    (roc_auc_score(y, blend50_oof), blend50_pred, "50/50 HC+Stack"),
]
v8_best_auc, v8_best_pred, v8_best_name = max(v7_cands, key=lambda x: x[0])
print(f"v8 internal best: {v8_best_name}  OOF={v8_best_auc:.5f}")
print(f"Weights: { {n: round(float(w), 3) for n, w in zip(_names, bw)} }")

test_fe_ids = test_A['id'].values if 'id' in test_A.columns else TEST_IDS
pred_df     = pd.DataFrame({'id': test_fe_ids, 'PitNextLap': np.clip(v8_best_pred, 0.001, 0.999)})
sub_out     = sub[['id']].merge(pred_df, on='id', how='left')
assert sub_out.shape[1] == 2 and sub_out['PitNextLap'].isna().sum() == 0
sub_out.to_csv(f"{OUT}submission_v8_solo.csv", index=False)
print(f"submission_v8_solo.csv saved  (OOF={v8_best_auc:.5f})")
print(f"\n{'Model':14s} OOF AUC")
for n, a in zip(_names+['Stack v8','v8 best'], [lgb_auc,xgb_auc,cb_auc,mlp_A_auc,mlp_B_auc,stack_auc,v8_best_auc]):
    print(f"  {n:14s}: {a:.5f}")


---
## 5-Way Blend: v8 + svanik-dataset + Public Notebooks

Public notebooks add diversity. The DE-anchored blend keeps your_v8 weight ≥ 0.25.


In [ ]:
N = len(sub)

def load_sub(path, label):
    df     = pd.read_csv(path)
    merged = sub[['id']].merge(df[['id','PitNextLap']], on='id', how='left')
    nans   = merged['PitNextLap'].isna().sum()
    if nans > 0: merged['PitNextLap'] = merged['PitNextLap'].fillna(0.2)
    p = merged['PitNextLap'].values.astype(np.float64)
    print(f"  {label:40s} | mean={p.mean():.4f} | std={p.std():.4f}")
    return p

print("Loading public submissions...")
pub_subs = {}
for root in NB_ROOTS:
    matches = glob.glob(f"{root}/**/*.csv", recursive=True)
    sub_matches = [m for m in matches if 'submission' in os.path.basename(m).lower()]
    if not sub_matches:
        for m in matches:
            try:
                df = pd.read_csv(m, nrows=3)
                if 'id' in df.columns and 'PitNextLap' in df.columns:
                    sub_matches.append(m); break
            except: pass
    if sub_matches:
        nm = root.split('/')[-1][:35]
        p_cand = load_sub(sorted(sub_matches)[-1], nm)
        if p_cand.max() < 0.85:
            print(f"  {nm}: max={p_cand.max():.3f} < 0.85, SKIPPING")
        elif any(np.corrcoef(rankdata(p_cand), rankdata(e))[0,1] > 0.9999 for e in pub_subs.values()):
            print(f"  {nm}: duplicate, SKIPPING")
        else:
            pub_subs[nm] = p_cand; print(f"  {nm}")
    else:
        print(f"  {root.split('/')[-1][:35]}: not found")

if os.path.exists(SVANIK_PATH):
    pub_subs['svanik-dataset'] = load_sub(SVANIK_PATH, 'svanik-dataset (LB 0.953)')
    print(f"  svanik-dataset OK")
else:
    print(f"  svanik-dataset not found at {SVANIK_PATH}")

your_v8_pred = sub_out['PitNextLap'].values.astype(np.float64)
blend_subs  = {'your_v8': your_v8_pred, **pub_subs}
blend_names = list(blend_subs.keys())
blend_preds = [blend_subs[n] for n in blend_names]
ranks_list  = [rankdata(p) / N for p in blend_preds]
corr_matrix = np.corrcoef(ranks_list)
_RM = np.stack(ranks_list, axis=0)

print(f"\nTotal in blend: {len(blend_names)}: {blend_names}")
triu = np.triu_indices(len(blend_names), 1)
for i, j in zip(*triu):
    c = corr_matrix[i, j]
    tag = "IDENTICAL" if c > 0.9999 else ("DIVERSE" if c < 0.97 else "")
    print(f"  {blend_names[i][:22]:22s} <-> {blend_names[j][:22]:22s}  {c:.4f}  {tag}")


---
## Optimized Blend Methods

In [ ]:
import time as _time

n_blend  = len(blend_names)
your_idx = blend_names.index('your_v8') if 'your_v8' in blend_names else 0
EPS      = 1e-9

KNOWN_LB = {
    'your_v8':                              float(v8_best_auc),
    'svanik-dataset':                       0.9530,
    'ps6e5-ensemble-0-95314-best-scor':     0.9531,
    'ps-s6-e5-realmlp-pytabkit':            0.9510,
    'pit-or-stay-f1-strategy-1':            0.9510,
}

_v7  = ranks_list[your_idx] - ranks_list[your_idx].mean()
_v7n = _v7 / (np.linalg.norm(_v7) + EPS)

def neg_oof_anchored(log_w):
    w = np.exp(log_w); w /= w.sum()
    blend  = w @ _RM
    b_c    = blend - blend.mean()
    corr   = np.dot(b_c, _v7n) / (np.linalg.norm(b_c) + EPS)
    div    = 1.0 - corr
    anchor = max(0.0, 0.25 - w[your_idx]) * 20.0
    conc   = max(0.0, w.max() - 0.70) * 10.0
    return -(float(v8_best_auc) + 0.008 * div) + anchor + conc

log_w0  = np.log(np.ones(n_blend) / n_blend)
bounds  = [(log_w0[i]-2.5, log_w0[i]+2.5) for i in range(n_blend)]

_t0 = _time.time()
de_result = differential_evolution(neg_oof_anchored, bounds, seed=SEED, maxiter=80, tol=1e-4, polish=False, popsize=8, mutation=(0.5,1.2), recombination=0.7)
w_de      = np.exp(de_result.x); w_de /= w_de.sum()
blend_de  = w_de @ _RM
print(f"DE done in {_time.time()-_t0:.1f}s")
print(f"Weights: {dict(zip([n[:20] for n in blend_names], w_de.round(3)))}")

best_nm = None
for w0_raw in [np.ones(n_blend)/n_blend, np.array([0.40]+[0.60/max(1,n_blend-1)]*(n_blend-1))]:
    w0_raw = w0_raw[:n_blend]; w0_raw /= w0_raw.sum()
    r = minimize(neg_oof_anchored, x0=np.log(w0_raw+EPS), method='Nelder-Mead', options={'maxiter':800,'xatol':1e-5,'fatol':1e-5})
    if best_nm is None or r.fun < best_nm.fun: best_nm = r
w_nm     = np.exp(best_nm.x); w_nm /= w_nm.sum()
blend_nm = w_nm @ _RM

blend_geo = np.exp(np.mean([np.log(r+EPS) for r in ranks_list], axis=0))
blend_geo = rankdata(blend_geo) / N
blend_eq  = np.mean(ranks_list, axis=0)

scores_arr = np.array([KNOWN_LB.get(n, 0.951) for n in blend_names])
try:
    R_inv  = np.linalg.inv(corr_matrix + 1e-5 * np.eye(n_blend))
    w_bay  = np.maximum(R_inv @ scores_arr, 0)
    w_bay  /= w_bay.sum() if w_bay.sum() > 1e-9 else 1
    blend_bay = w_bay @ _RM
except:
    blend_bay = blend_eq

all_blends = {'de_anchored': blend_de, 'nm_anchored': blend_nm,
              'geometric_mean': blend_geo, 'equal_rank': blend_eq, 'bayesian': blend_bay}

print(f"\n{'Method':35s}  std")
for bname, b in all_blends.items():
    print(f"  {bname:33s}  {b.std():.5f}")

PRIMARY_BLEND = blend_de; PRIMARY_BLEND_NAME = 'de_anchored'


---
## Save Submissions

In [ ]:
for bname, b in all_blends.items():
    out_df = sub[['id']].copy(); out_df['PitNextLap'] = np.clip(b, 0.001, 0.999)
    out_df.to_csv(f"{OUT}submission_{bname[:35]}.csv", index=False)

main_out = sub[['id']].copy(); main_out['PitNextLap'] = np.clip(PRIMARY_BLEND, 0.001, 0.999)
assert main_out.shape == (len(sub), 2) and main_out['PitNextLap'].isna().sum() == 0
main_out.to_csv(f"{OUT}submission.csv", index=False)

print(f"submission.csv saved = {PRIMARY_BLEND_NAME}")
print(f"\nFINAL REPORT")
print(f"{'='*50}")
for n, a in zip(_names+['Stack v8','v8 best'], [lgb_auc,xgb_auc,cb_auc,mlp_A_auc,mlp_B_auc,stack_auc,v8_best_auc]):
    print(f"  {n:14s}: {a:.5f}")
print(f"\n  Blend sources: {blend_names}")
print(f"  Primary:       {PRIMARY_BLEND_NAME}")
print(f"  Expected LB:   {v8_best_auc:.5f} (solo) -> {v8_best_auc+0.002:.5f}+ (blended)")

import subprocess
result = subprocess.run(
    ['kaggle', 'competitions', 'submit',
     '-c', 'playground-series-s6e5',
     '-f', f'{OUT}submission.csv',
     '-m', f'v8 {PRIMARY_BLEND_NAME} OOF={v8_best_auc:.5f}'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
